# distributed-sampler-shard — worked example 1: Verify that DistributedSampler shards are disjoint and their union covers the dataset

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `distributed-sampler-shard`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`DistributedSampler` assigns a disjoint subset of dataset indices to each rank so that during one epoch, no two ranks process the same sample. When `shuffle=True`, padding may add a few repeated indices to make all shards equal length. The union of all rank shards (with duplicates removed) should equal the full index set `{0, ..., len(dataset)-1}`.

## Worked solution

Consider a dataset of 10 items and `world_size=3`. Each shard has `ceil(10/3) = 4` indices (with padding).

**Step 1.** Build a sampler for each rank, call `set_epoch(0)`, collect indices.

**Step 2.** Check disjointness: pairwise, no two shards share any index (before padding). Since `10 % 3 != 0`, exactly two ranks get a padded extra sample — but the padded index is a *repeated* index, not a unique new one, so pairwise disjointness holds on the true slice.

**Step 3.** Check coverage: `set(chain(*shards)) == set(range(10))`. Every dataset index appears at least once across all shards.

**Why `set_epoch` matters:** it seeds the shuffle. Without it, every epoch uses the same permutation — each rank sees the same samples every epoch, defeating the purpose of shuffling.

In [ ]:
import torch
from torch.utils.data import TensorDataset
from torch.utils.data.distributed import DistributedSampler
from itertools import chain

def collect_and_verify_shards(n_samples, world_size, seed=42):
    dataset = TensorDataset(torch.arange(n_samples))

    shards = []
    for rank in range(world_size):
        sampler = DistributedSampler(
            dataset, num_replicas=world_size, rank=rank,
            shuffle=True, seed=seed
        )
        sampler.set_epoch(0)
        shards.append(list(sampler))

    # Each shard has the same length
    shard_len = len(shards[0])
    assert all(len(s) == shard_len for s in shards), 'Shards have unequal length'
    print(f'Shard length per rank: {shard_len}')

    # Union covers all dataset indices
    all_indices = set(chain(*shards))
    assert all_indices == set(range(n_samples)), 'Union of shards does not cover full dataset'
    print(f'Union of shards covers all {n_samples} indices: True')

    # Pairwise disjoint (ignoring padding)
    for i in range(world_size):
        for j in range(i + 1, world_size):
            overlap = set(shards[i]) & set(shards[j])
            # Overlap is only allowed due to padding (repeated indices)
            # True (non-padded) shard size
            true_size = n_samples // world_size
            print(f'  Ranks {i},{j} overlap size: {len(overlap)} (from padding only if > 0)')
    return shards

shards = collect_and_verify_shards(n_samples=13, world_size=4, seed=7)
print('Shard indices per rank:')
for i, s in enumerate(shards):
    print(f'  Rank {i}: {sorted(s)}')